In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense
import torch  # Added import for torch.cuda usage

!pip install ta
import ta

# Create the saving directory if it doesn't exist
os.makedirs("models2", exist_ok=True)

# Verify GPU availability via TensorFlow.
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    print(f"TensorFlow GPU detected: {physical_devices}")
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("No TensorFlow GPU detected. Running on CPU.")

# Verify GPU availability using torch.cuda.
if torch.cuda.is_available():
    print(f"torch.cuda is available: {torch.cuda.get_device_name(0)}")
else:
    print("torch.cuda is not available. Running on CPU.")

# Function to calculate 3-day returns.
def calculate_returns(prices, horizon):
    # Returns computed as: price[t+horizon] / price[t] - 1
    returns = prices[horizon:] / prices[:-horizon] - 1
    return returns

# Function to create a dataset for LSTM when predicting returns.
# Assumes that features and target returns are aligned.
def create_dataset_returns(features, target_returns, lookback=30):
    X, y = [], []
    # For a given time index i, we use the previous 'lookback' days of features
    # as input and the corresponding return as the label.
    for i in range(lookback, len(features)):
        X.append(features[i - lookback:i])
        y.append(target_returns[i, 0])
    return np.array(X), np.array(y)

# Function to build an LSTM model.
def build_lstm_model(lookback, n_features):
    model = Sequential()
    model.add(LSTM(100, return_sequences=True, input_shape=(lookback, n_features)))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dense(25))
    model.add(Dense(1))  # Predict a single return value
    # Using the default Adam learning rate (0.001)
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Function to add technical indicators to a DataFrame.
def add_technical_indicators(df, column_name='prices_close'):
    df = df.copy()
    close = df[column_name]
    df['SMA_10'] = ta.trend.SMAIndicator(close, window=10).sma_indicator()
    df['EMA_10'] = ta.trend.EMAIndicator(close, window=10).ema_indicator()
    df['RSI_14'] = ta.momentum.RSIIndicator(close, window=14).rsi()
    df['Momentum_10'] = ta.momentum.AwesomeOscillatorIndicator(high=close, low=close).awesome_oscillator()
    df['ATR_14'] = ta.volatility.AverageTrueRange(high=close, low=close, close=close, window=14).average_true_range()
    return df

# Load the stock data. The CSV is assumed to have dates as index and one column per stock.
stocks = pd.read_csv('/kaggle/input/prices/prices.csv', index_col=0)
stocks.index = pd.to_datetime(stocks.index)

# Parameters
lookback = 30       # number of past days to use as input
horizon = 3         # predict 3-day return
models_dict = {}
all_metrics = {}  # Dictionary for storing metrics

# Process only the first stock for demonstration.
for stock_name in stocks.columns[:13]:
    print(f"\nProcessing stock: {stock_name}")
    
    # Prepare the data: rename the column to 'prices_close' and add technical indicators.
    stock_data = stocks[[stock_name]].rename(columns={stock_name: 'prices_close'})
    stock_data = add_technical_indicators(stock_data)
    stock_data.dropna(inplace=True)
    
    # Compute the 3-day returns from the price.
    prices = stock_data['prices_close'].values.reshape(-1, 1)
    returns = calculate_returns(prices, horizon)  # shape = (n - horizon, 1)
    
    # Extract features in the order: price, SMA, EMA, RSI, Momentum, ATR.
    features = stock_data[['prices_close', 'SMA_10', 'EMA_10', 'RSI_14', 'Momentum_10', 'ATR_14']].values
    # Align features with returns by dropping the last "horizon" rows.
    features_aligned = features[:-horizon]
    dates = stock_data.index[:-horizon]
    
    # Set up rolling training/testing periods.
    training_dates = []
    testing_dates = []
    start_date = dates[0]
    end_date = dates[-1]
    while start_date + pd.DateOffset(years=4) <= end_date:
        training_dates.append((start_date, start_date + pd.DateOffset(years=3)))
        testing_dates.append((start_date + pd.DateOffset(years=3), start_date + pd.DateOffset(years=4)))
        start_date += pd.DateOffset(years=1)
    
    # Loop over each training/testing period.
    for train_date, test_date in zip(training_dates, testing_dates):
        # Create boolean masks based on dates.
        train_mask = (dates >= train_date[0]) & (dates <= train_date[1])
        test_mask = (dates >= test_date[0]) & (dates <= test_date[1])
        
        train_features = features_aligned[train_mask]
        test_features  = features_aligned[test_mask]
        train_returns = returns[train_mask]
        test_returns  = returns[test_mask]
        
        # Scale features and returns separately.
        features_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_train_features = features_scaler.fit_transform(train_features)
        scaled_test_features = features_scaler.transform(test_features)
        
        returns_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_train_returns = returns_scaler.fit_transform(train_returns)
        scaled_test_returns = returns_scaler.transform(test_returns)
        
        # Create LSTM datasets using a lookback window.
        X_train, y_train = create_dataset_returns(scaled_train_features, scaled_train_returns, lookback)
        X_test, y_test = create_dataset_returns(scaled_test_features, scaled_test_returns, lookback)
        
        n_features = X_train.shape[2]
        
        # Build and train the LSTM model.
        model = build_lstm_model(lookback, n_features)
        history = model.fit(X_train, y_train, batch_size=32, epochs=100,
                            validation_data=(X_test, y_test), verbose=0)
        
        # Save the model and scalers.
        model_name = f"{stock_name}_{train_date[0].strftime('%Y%m%d')}_{test_date[0].strftime('%Y%m%d')}"
        model.save(f"models2/{model_name}.h5")
        joblib.dump({'features_scaler': features_scaler, 'returns_scaler': returns_scaler},
                    f"models2/{model_name}_scalers.pkl")
        models_dict[model_name] = (model, features_scaler, returns_scaler)
        print(f"Model and scalers saved: {model_name}")
        
        # Evaluate the model: predict returns on the test set.
        predictions_scaled = model.predict(X_test)
        predictions = returns_scaler.inverse_transform(predictions_scaled)
        y_test_actual = returns_scaler.inverse_transform(y_test.reshape(-1, 1))
        
        # Calculate RMSE of the predicted returns.
        rmse = np.sqrt(mean_squared_error(y_test_actual, predictions))
        print(f"{model_name} - RMSE: {rmse:.4f}")
        
        # Calculate MAPE (Mean Absolute Percentage Error).
        # epsilon = 1e-10  # small constant to avoid division by zero
        # mape = np.mean(np.abs((y_test_actual - predictions) / np.where(np.abs(y_test_actual) < epsilon, epsilon, y_test_actual))) * 100
        # print(f"{model_name} - MAPE: {mape:.4f}%")
        
        # Store metrics.
        all_metrics[model_name] = {
            'train_mse': mean_squared_error(y_train, model.predict(X_train)),
            'test_mse': mean_squared_error(y_test, predictions_scaled),
            'train_rmse': np.sqrt(mean_squared_error(y_train, model.predict(X_train))),
            'test_rmse': rmse,
            'train_mape': np.mean(np.abs((returns_scaler.inverse_transform(model.predict(X_train)) - returns_scaler.inverse_transform(y_train.reshape(-1, 1)))
                                        / np.where(np.abs(returns_scaler.inverse_transform(y_train.reshape(-1, 1))) < epsilon, epsilon, returns_scaler.inverse_transform(y_train.reshape(-1, 1))))) * 100,
            'test_mape': mape
        }
        
        print(f"{model_name} Metrics:")
        print(f"  Train MSE: {all_metrics[model_name]['train_mse']:.4f} | RMSE: {all_metrics[model_name]['train_rmse']:.4f} | %")
        print(f"  Test  MSE: {all_metrics[model_name]['test_mse']:.4f} | RMSE: {all_metrics[model_name]['test_rmse']:.4f} | %")
        
        # Plot predicted vs. actual returns on the training set.
        plt.figure(figsize=(12, 6))
        train_dates_aligned = dates[train_mask]
        train_preds = model.predict(X_train)
        plt.plot(train_dates_aligned[lookback:], returns_scaler.inverse_transform(y_train.reshape(-1, 1)), label='Actual 3-Day Returns')
        plt.plot(train_dates_aligned[lookback:], returns_scaler.inverse_transform(train_preds), label='Predicted 3-Day Returns')
        plt.title(f"{stock_name} 3-Day Return Prediction on Training Set\n({train_date[0].strftime('%Y%m%d')} to {train_date[1].strftime('%Y%m%d')})")
        plt.xlabel('Date')
        plt.ylabel('Return')
        plt.legend()
        plt.grid()
        plt.show()
        
        # Plot predicted vs. actual returns on the test set.
        plt.figure(figsize=(12, 6))
        test_dates_aligned = dates[test_mask]
        plt.plot(test_dates_aligned[lookback:], y_test_actual, label='Actual 3-Day Returns')
        plt.plot(test_dates_aligned[lookback:], predictions, label='Predicted 3-Day Returns')
        plt.title(f"{stock_name} 3-Day Return Prediction on Test Set\n({test_date[0].strftime('%Y%m%d')} to {test_date[1].strftime('%Y%m%d')})")
        plt.xlabel('Date')
        plt.ylabel('Return')
        plt.legend()
        plt.grid()
        plt.show()


In [17]:
import os
import json


# Save the metrics dictionary to a JSON file
with open("/kaggle/working/models2/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=4)


In [18]:
stocks[:13]

,NTPC,PWGR,TCS,WPRO,DLFU,GCPL,LT,SIEM,SUNP,CIPLA,HDFCB,ICICIBC,RELIANCE,ONGC,HUVR,ITC,BHARTI,IDEA,SAIL,HNDL
Date,,,,,,,,,,,,,,,,,,,,
2015-01-01,118.71,77.12,1272.78,103.58,137.40,321.97,1001.97,906.70,822.20,628.40,476.03,320.27,200.33,229.37,758.45,231.55,327.07,96.25,82.75,158.45
2015-01-02,120.33,78.05,1289.72,104.49,139.20,323.37,1023.10,914.35,826.25,630.15,482.65,329.36,199.80,232.80,755.95,232.28,329.00,96.58,82.85,160.10
2015-01-05,120.00,78.13,1270.13,104.68,135.80,323.48,1037.33,918.65,826.75,633.00,478.58,330.05,197.61,235.43,760.30,233.19,321.70,92.74,82.70,156.85
2015-01-06,116.00,76.39,1223.30,102.23,133.65,317.33,1002.73,895.70,808.55,614.35,471.13,316.05,188.64,222.10,774.70,227.20,319.18,90.90,78.65,153.40
2015-01-07,118.79,77.48,1208.85,101.46,134.60,320.05,1000.33,899.20,809.80,611.95,472.50,307.50,192.75,225.37,801.90,222.98,319.95,90.45,77.80,148.90
2015-01-08,120.79,77.82,1221.90,102.19,142.75,330.05,1006.70,920.40,818.80,618.20,482.43,315.86,189.98,227.87,817.05,228.56,325.49,91.99,79.15,152.55
2015-01-09,116.88,77.88,1256.15,103.78,137.85,341.18,1000.17,914.45,830.05,632.35,487.83,310.77,194.10,234.00,864.60,225.09,320.67,89.15,78.55,155.00
2015-01-12,116.67,76.78,1254.85,104.26,136.50,352.95,1022.80,926.20,835.30,630.75,483.53,315.05,191.84,231.80,896.60,225.69,314.49,88.24,80.00,151.10
2015-01-13,115.79,76.42,1248.95,105.66,133.25,360.38,1013.00,919.40,833.10,636.40,481.68,309.91,190.23,226.63,884.55,227.11,312.11,88.79,78.25,151.70
